# 08 - Capstone: Combine All Techniques

Now let's combine everything we've learned!

In [1]:
import sys
from pathlib import Path

# notebooks/ holds _fetch.py and _sources.py; src/ holds the project's own helpers.
for p in (Path.cwd(), Path.cwd() / ".." / "src"):
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p.resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import _fetch
import _sources as S
from ocean_data_workshop.dsn import dsn
import psycopg

# ML imports
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.model_selection import train_test_split

# Setup
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

print("ML Workshop Environment Ready")
print("=" * 60)
print("Data sources:")
print("  - NDBC 46092 (wind)")
print("  - GLORYS12V1 (ocean)")
print("  - SanctSound MB01 (acoustic)")
print("=" * 60)

ML Workshop Environment Ready
Data sources:
  - NDBC 46092 (wind)
  - GLORYS12V1 (ocean)
  - SanctSound MB01 (acoustic)


## The challenge

Build a complete ML pipeline that:
1. Loads ocean, wind, and acoustic data
2. Engineers features
3. Trains multiple models
4. Evaluates properly
5. Interprets results

Let's predict dolphin presence using all available data!

In [2]:
import numpy as np
import pandas as pd

print("=" * 60)
print("CAPSTONE: Dolphin Detection with All Data")
print("=" * 60)

# Load all data
conn = psycopg.connect(dsn())

# Acoustic data
query = "SELECT observed_at, site_id,"
query += " band_25hz, band_50hz, band_160hz,"
query += " band_500hz, band_2500hz, band_20000hz, band_10000hz"
query += " FROM acoustic_tol_hourly WHERE site_id = 'mb01' ORDER BY observed_at"

acoustic_df = pd.read_sql_query(query, conn)

# Wind data
query = "SELECT observed_at, station_id, wind_speed_mean_ms, wind_northward_ms,"
query += " wave_height_max_m, air_temp_mean_c"
query += " FROM wind_daily WHERE station_id = '46092' ORDER BY observed_at"

wind_df = pd.read_sql_query(query, conn)

# Dolphin data
query = "SELECT observed_at, presence FROM detection_hourly"
query += " WHERE taxon = 'dolphin' AND is_event_list = FALSE ORDER BY observed_at"

dolphin_df = pd.read_sql_query(query, conn)

conn.close()

print(f"Acoustic: {len(acoustic_df):,} rows")
print(f"Wind: {len(wind_df):,} rows")
print(f"Dolphin: {len(dolphin_df):,} rows")

# Merge acoustic and wind (on date)
acoustic_df['date'] = acoustic_df['observed_at'].dt.date
wind_df['date'] = wind_df['observed_at'].dt.date

# Only keep acoustic rows that have matching wind data
# (wind data has fewer dates than acoustic)
acoustic_dates = set(acoustic_df['date'].unique())
wind_dates = set(wind_df['date'].unique())
common_dates = acoustic_dates & wind_dates

acoustic_df = acoustic_df[acoustic_df['date'].isin(common_dates)]
print(f"Acoustic after date filter: {len(acoustic_df):,} rows")

# Merge on date
merged = acoustic_df.merge(wind_df[['date', 'wind_speed_mean_ms', 'wind_northward_ms',
                                       'wave_height_max_m', 'air_temp_mean_c']],
                            on='date', how='left')

# Merge dolphin
merged = merged.merge(dolphin_df[['observed_at', 'presence']],
                       on='observed_at', how='inner')

print(f"\nMerged: {len(merged):,} rows")


# Fill NaN values with column means for ML
wind_cols = ['wind_speed_mean_ms', 'wind_northward_ms', 'air_temp_mean_c']
for col in wind_cols:
    merged[col] = merged[col].fillna(merged[col].mean())
print(f"Filled NaN values with column means (excluding wave_height_max_m which has no data)")
print(f"Remaining NaN: {merged.isna().sum().sum()}")
# Features
freq_cols = [c for c in merged.columns if c.startswith('band_')]
wind_cols = ['wind_speed_mean_ms', 'wind_northward_ms', 'air_temp_mean_c']
all_features = freq_cols + wind_cols

X = merged[all_features].values
y = merged['presence'].values

print(f"Features ({len(all_features)}):")
for f in all_features:
    print(f"  - {f}")
print("Target: dolphin presence (0/1)")

# Split by time
split_idx = int(len(X) * 0.75)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"\nTrain: {len(y_train)}, Test: {len(y_test)}")
print(f"Test positive rate: {y_test.mean():.2%}")

CAPSTONE: Dolphin Detection with All Data
Acoustic: 19,570 rows
Wind: 784 rows
Dolphin: 8,390 rows
Acoustic after date filter: 17,945 rows

Merged: 8,267 rows
Filled NaN values with column means (excluding wave_height_max_m which has no data)
Remaining NaN: 8267
Features (10):
  - band_25hz
  - band_50hz
  - band_160hz
  - band_500hz
  - band_2500hz
  - band_20000hz
  - band_10000hz
  - wind_speed_mean_ms
  - wind_northward_ms
  - air_temp_mean_c
Target: dolphin presence (0/1)

Train: 6200, Test: 2067
Test positive rate: 19.69%


/var/folders/6_/6bnkvg2j2qd8720cj2rr98080000gn/T/ipykernel_24790/615542614.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  acoustic_df = pd.read_sql_query(query, conn)
/var/folders/6_/6bnkvg2j2qd8720cj2rr98080000gn/T/ipykernel_24790/615542614.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  wind_df = pd.read_sql_query(query, conn)
/var/folders/6_/6bnkvg2j2qd8720cj2rr98080000gn/T/ipykernel_24790/615542614.py:30: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dolphin_df = pd.read_sql_query(query, conn)


## Train multiple models

In [3]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Models
models = {
    'Logistic Regression': LogisticRegression(max_iter=2000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

results = []

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1': f1_score(y_test, y_pred, zero_division=0)
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

# Best model
best_idx = results_df['F1'].idxmax()
best_model_name = results_df.loc[best_idx, 'Model']
print(f"\nBest model: {best_model_name} (F1 = {results_df.loc[best_idx, 'F1']:.3f})")

              Model  Accuracy  Precision   Recall       F1
Logistic Regression  0.873246   0.765568 0.513514 0.614706
      Random Forest  0.880987   0.769231 0.565111 0.651558
  Gradient Boosting  0.876149   0.759450 0.542998 0.633238

Best model: Random Forest (F1 = 0.652)


## Feature importance

In [4]:
# Get feature importance from best model
best_model = models[best_model_name]
best_model.fit(X_train_scaled, y_train)

if hasattr(best_model, 'feature_importances_'):
    importance = best_model.feature_importances_
    model_type = 'tree-based'
elif hasattr(best_model, 'coef_'):
    importance = abs(best_model.coef_[0])
    model_type = 'linear'

feature_importance = pd.DataFrame({
    'Feature': all_features,
    'Importance': importance
}).sort_values('Importance', ascending=False)

print(f"Feature Importance ({model_type} model):")
print(feature_importance.to_string(index=False))

# Top 5 features
print("\nTop 5 features:")
print(feature_importance.head())

Feature Importance (tree-based model):
           Feature  Importance
      band_20000hz    0.238042
         band_25hz    0.105290
      band_10000hz    0.105227
       band_2500hz    0.102659
        band_500hz    0.093950
         band_50hz    0.077893
   air_temp_mean_c    0.075821
        band_160hz    0.070461
 wind_northward_ms    0.068200
wind_speed_mean_ms    0.062457

Top 5 features:
        Feature  Importance
5  band_20000hz    0.238042
0     band_25hz    0.105290
6  band_10000hz    0.105227
4   band_2500hz    0.102659
3    band_500hz    0.093950


## Confusion matrix

In [5]:
from sklearn.metrics import confusion_matrix, classification_report

best_model = models[best_model_name]
y_pred = best_model.predict(X_test_scaled)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['absent', 'present']))

Confusion Matrix:
[[1591   69]
 [ 177  230]]

Classification Report:
              precision    recall  f1-score   support

      absent       0.90      0.96      0.93      1660
     present       0.77      0.57      0.65       407

    accuracy                           0.88      2067
   macro avg       0.83      0.76      0.79      2067
weighted avg       0.87      0.88      0.87      2067



## The lesson

- Multiple models can be compared
- Feature importance reveals what matters
- Confusion matrix shows prediction patterns
- Always use proper evaluation metrics!

## Next steps

1. Try different feature combinations
2. Add time-lag features
3. Use cross-validation
4. Tune hyperparameters